# 3.1 Model and Context - Model 配置与使用

- **对应文档**: [task_model](https://doc.agentscope.io/tutorial/task_model.html)
- **本讲目标**: 掌握 AgentScope 中 Model 的配置、多模型切换与 API 封装。
- **前置知识**: 02_react_agent。

## 学习要点

- 不同后端（如 DashScope、OpenAI 等）的 Model 封装。
- stream / 非 stream、enable_thinking 等参数。

In [1]:
# 在此按 doc 编写 Model 配置示例
# from agentscope.model import DashScopeChatModel
# 参考 https://doc.agentscope.io/tutorial/task_model.html
print("请参考 task_model 文档完成 Model 示例代码")


请参考 task_model 文档完成 Model 示例代码


1. 非流式返回时，返回类型是 ChatResponse 实例；

In [2]:
import asyncio
import json
import os

from agentscope.message import TextBlock, ToolUseBlock, ThinkingBlock, Msg
from agentscope.model import ChatResponse, DashScopeChatModel

async def example_model_call() -> None:
    """使用 DashScopeChatModel 的示例。"""
    model = DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
    )

    res = await model(
        messages=[
            {"role": "user", "content": "你好！"},
        ],
    )

    # 您可以直接使用响应内容创建 ``Msg`` 对象
    msg_res = Msg("Friday", res.content, "assistant")

    print("LLM 返回结果:", res)
    print("作为 Msg 的响应:", msg_res)


await example_model_call()

LLM 返回结果: ChatResponse(content=[{'type': 'text', 'text': '你好！有什么我可以帮助你的吗？'}], id='2026-03-13 17:00:37.383_bef01a', created_at='2026-03-13 17:00:37.383', type='chat', usage=ChatUsage(input_tokens=10, output_tokens=8, time=2.875892, type='chat', metadata=GenerationUsage(input_tokens=10, output_tokens=8)), metadata=None)
作为 Msg 的响应: Msg(id='R4rEYtAxnSooJq2ETW5xSJ', name='Friday', content=[{'type': 'text', 'text': '你好！有什么我可以帮助你的吗？'}], role='assistant', metadata={}, timestamp='2026-03-13 17:00:37.383', invocation_id='None')


2. 流式返回时，返回的是 ChatResponse 的异步生成器

In [9]:
async def example_streaming() -> None:
    """使用流式模型的示例。"""
    model = DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=True,
    )

    generator = await model(
        messages=[
            {
                "role": "user",
                "content": "从 1 数到 20，只报告数字，不要任何其他信息。",
            },
        ],
    )
    print("响应的类型:", type(generator))

    async for chunk in generator:
        print(f"\t{chunk.content[0].pop('text')}\n")

await example_streaming()

响应的类型: <class 'async_generator'>
	1

	1 2 

	1 2 3 4

	1 2 3 4 5 

	1 2 3 4 5 6 7 8 

	1 2 3 4 5 6 7 8 9 10 1

	1 2 3 4 5 6 7 8 9 10 11 12 1

	1 2 3 4 5 6 7 8 9 10 11 12 13 14 1

	1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 1

	1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 1

	1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20

	1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20



3. 推理模型

In [11]:
async def example_reasoning() -> None:
    """使用推理模型的示例。"""
    model = DashScopeChatModel(
        model_name="qwen-turbo",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        enable_thinking=True,
    )

    res = await model(
        messages=[
            {"role": "user", "content": "我是谁？"},
        ],
    )

    last_chunk = None
    async for chunk in res:
        last_chunk = chunk
    print("最终响应:")
    print(last_chunk)
    print("finish!")


await example_reasoning()

最终响应:
ChatResponse(content=[{'type': 'thinking', 'thinking': '好的，用户问“我是谁？”。这个问题看起来简单，但其实很深奥。首先，我需要考虑用户为什么会问这个问题。可能是在进行自我反思，或者对自我身份感到困惑。也有可能是在哲学层面思考存在主义的问题。\n\n接下来，我要分析用户可能的背景。他们可能是个学生，正在学习哲学或心理学；或者是一个普通人，突然对自我产生疑问。也有可能是在某种情绪低落的时候，寻求答案。\n\n然后，我需要确定用户的需求。他们可能想要一个具体的答案，或者希望得到一些指导来自己探索。也有可能只是想聊聊，寻找共鸣。\n\n考虑到用户可能没有明确表达的深层需求，比如对自我认同的困惑、存在感的缺失，或者对人生意义的追寻。这时候需要提供一些引导性的思考，而不是直接给出答案。\n\n还要注意避免过于学术化的回答，保持自然和亲切。可能需要结合心理学、哲学和日常生活的例子，让用户更容易理解和接受。\n\n另外，要确保回答的结构清晰，分点说明不同的角度，比如生物学、心理学、哲学和社会角色，这样用户可以从中找到共鸣或启发。\n\n最后，要鼓励用户继续探索，因为“我是谁”这个问题没有标准答案，每个人都有自己的答案。同时，保持开放和包容的态度，让用户感到被理解和支持。\n'}, {'type': 'text', 'text': '“我是谁？”这个问题看似简单，却可能引发深刻的思考。它可能涉及哲学、心理学、社会角色等多个层面。以下是一些可能的视角，或许能帮助你探索答案：\n\n---\n\n### 1. **生物学层面**  \n从科学的角度看，你是由细胞、基因、神经元等组成的生物体，是宇宙中独特的存在。你的身体、大脑和基因决定了你与他人不同的生理特征。但“我是谁”远不止于此。\n\n---\n\n### 2. **心理学层面**  \n心理学认为，“我”是自我意识的产物。你通过记忆、情感、思想和经验构建出对“自己”的认知。比如：  \n- **身份认同**：你可能认为自己是“一个学生”“一个父母”“一个艺术家”，这些标签是社会和自我的共同塑造。  \n- **内在冲突**：有时你会感到“我既想自由，又渴望稳定”，这种矛盾可能源于不同价值观的拉扯。  \n- **成长变化**：随着经历改变，“我”也在不断更新——